In [0]:
# Traditional partitioning works better with lower-cardinality columns and sufficiently large data per partition—not every commonly filtered column deserves to become a partition column.

# Before liquid clustering, the selective lookup for customer_key = 87421 returned only 50 rows, but Databricks had to read 15.73 MB across 8 tasks, completing in 699 ms. 
# After clustering the table by customer_key and running OPTIMIZE, the same query still returned 50 rows but read only 11.92 MB using 1 task, reducing data scanned by approximately 24.2%. 
# The runtime increased to 1.399 s, which shows why execution time alone is not a reliable optimization metric on a small ~16 MB benchmark table: compute/serverless overhead and runtime variability can dominate at this scale. The more meaningful result is the reduction in bytes scanned and tasks, demonstrating that clustering improved the physical organization of data and allowed better data skipping for customer-key lookups. At larger production scale, reducing unnecessary I/O becomes increasingly valuable.

In [0]:
%sql
DESCRIBE DETAIL sentinel_dev.gold.fact_orders;

In [0]:
%sql
DESCRIBE HISTORY sentinel_dev.gold.fact_orders;

In [0]:
%sql
SELECT
    version,
    timestamp,
    operation,
    operationParameters
FROM (
    DESCRIBE HISTORY sentinel_dev.gold.fact_orders
)
ORDER BY version DESC;

In [0]:
%sql
EXPLAIN FORMATTED
SELECT
    order_date,
    SUM(total_amount) AS revenue
FROM sentinel_dev.gold.fact_orders
WHERE order_date >= current_date() - INTERVAL 30 DAYS
GROUP BY order_date;

In [0]:
%sql
EXPLAIN FORMATTED
SELECT
    order_date,
    SUM(total_amount) AS revenue
FROM sentinel_dev.gold.fact_orders_perf
WHERE order_date >= current_date() - INTERVAL 30 DAYS
GROUP BY order_date;

In [0]:
%sql
SELECT
    order_date,
    SUM(total_amount) AS revenue
FROM sentinel_dev.gold.fact_orders_perf
WHERE order_date >= current_date() - INTERVAL 30 DAYS
GROUP BY order_date
ORDER BY order_date;

-- Statement
-- Tasks
-- Duration
-- Rows read
-- Bytes read
-- Bytes written
-- Genie review
-- SELECT
--   order_date,
--   SUM(total_amount) AS revenue
-- FROM
--   sentinel_dev.gold.fact_orders_perf
-- WHERE
--   order_date >= current_date() - INTERVAL 30 DAYS
-- GROUP BY
--   order_date
-- ORDER BY
--   order_date
-- Aug 25, 2026, 01:26 PM
-- 9/9 completed
-- 1 s 847 ms
-- 424,669
-- 15.65 MB
-- 0 B



In [0]:
%sql
EXPLAIN FORMATTED
SELECT *
FROM sentinel_dev.gold.fact_orders_perf
WHERE customer_key = 87421;

In [0]:
%sql
SELECT *
FROM sentinel_dev.gold.fact_orders_perf
WHERE customer_key = 87421;


-- Statement
-- Tasks
-- Duration
-- Rows read
-- Bytes read
-- Bytes written
-- Genie review
-- SELECT * FROM sentinel_dev.gold.fact_orders_perf WHERE customer_key = 87421
-- Aug 25, 2026, 01:28 PM
-- 8/8 completed
-- 699 ms
-- 50
-- 15.73 MB
-- 0 B


In [0]:
%sql
-- Don't partition by customer_key
-- Databricks basically has to say:
-- “Customer 87421 could be in almost every file.”
-- That's exactly the kind of access pattern clustering can improve.

-- ADD LIQUID CLUSTERING
ALTER TABLE sentinel_dev.gold.fact_orders_perf
CLUSTER BY (customer_key);

In [0]:
%sql
OPTIMIZE sentinel_dev.gold.fact_orders_perf;

In [0]:
%sql
DESCRIBE DETAIL sentinel_dev.gold.fact_orders_perf;

In [0]:
%sql
SELECT *
FROM sentinel_dev.gold.fact_orders_perf
WHERE customer_key = 87421;

-- Statement
-- Tasks
-- Duration
-- Rows read
-- Bytes read
-- Bytes written
-- Genie review
-- SELECT * FROM sentinel_dev.gold.fact_orders_perf WHERE customer_key = 87421
-- Aug 25, 2026, 07:19 PM
-- 1/1 completed
-- 1 s 399 ms
-- 50
-- 11.92 MB
-- 0 B
